# Tiny-Aya Base Model Testing Notebook

TinyAya-Base, a pretrained 3.35B-parameter model: TinyAya-Base covers 70+ languages*, including many lower-resourced languages from around the globe.
[blog source](https://cohere.com/blog/cohere-labs-tiny-aya)


## Setup and Installation

In [2]:
!pip install -q transformers accelerate torch huggingface_hub

### Log in to HuggingFace account

In [3]:
from huggingface_hub import notebook_login
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


notebook_login()

In [4]:
import os
from google.colab import userdata


HF_TOKEN = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = HF_TOKEN

### Model Loading and Tokenizer

In [5]:
MODEL_NAME = "CohereLabs/tiny-aya-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME,device_map="auto")

print(f"Model loaded! Device: {model.device}")
print(f"Parameters: {model.num_parameters() / 1e9:.2f}B")

config.json:   0%|          | 0.00/1.96k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.26k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded! Device: cuda:0
Parameters: 3.35B


### Helper Function

In [6]:
def gen_res(prompt, max_new_tokens=128, temperature=0.1, top_p=0.95,top_k=50):

    inputs = tokenizer( prompt,return_tensors="pt").to(model.device)

    gen_tokens = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            top_k = 50
        )

    input_len = inputs["input_ids"].shape[-1]
    res = tokenizer.decode(gen_tokens[0][input_len:], skip_special_tokens=True)
    return res

#### Test

In [7]:
# input prompt
prompt = "The capital of Spain is"
# LLM's output
res = gen_res(prompt,top_k=1)
print(prompt) # display input prompt
print(res) # display LLM's output

The capital of Spain is
 ( ).

A. Madrid
B. Barcelona
C. Valencia
D. Seville

Answer: A


### Arithmetic

❌ Mistake

- right answer : `$13.7`

In [ ]:
prompt = "If you buy 3 apples for $6.3 each and pay with $20, how much change do you get?"
res = gen_res(prompt)
print(prompt)
print(res)

If you buy 3 apples for $6.3 each and pay with $20, how much change do you get?
 ( )

A. $2.7
B. $3.7
C. $4.7
D. $5.7

Answer: A


❌ Mistake


- right answer : `$2.42`

In [ ]:
prompt = "If you buy 3 apples for $7.26, how much does one apple cost?"
res = gen_res(prompt)
print(prompt)
print(res)

If you buy 3 apples for $7.26, how much does one apple cost?
 If you buy 3 apples for $7.26, how much does one apple cost?

A. $2.4
B. $2.6
C. $2.8
D. $3.0

Answer: B


✅ Right answer example

In [ ]:
# @title
# { display-mode: "form" }

prompt = "All cats are animals. Some animals are black. Can we conclude that some cats are black?"
res = gen_res(prompt)
print(prompt)
print(res)

All cats are animals. Some animals are black. Can we conclude that some cats are black?
 ()

A. Yes
B. No
C. Cannot be determined
D. None of the above

Answer: C


### Hallucination / Logical Error


❌ Mistake

- right answer : cannot be determined

In [ ]:
prompt = "Tom is bigger than Jack. Peter is smaller than Jack. What is the age of Jack ?"
res = gen_res(prompt)
print(prompt)
print(res)

Tom is bigger than Jack. Peter is smaller than Jack. What is the age of Jack ?

If Tom is bigger than Jack and Peter is smaller than Jack, then Jack must be the smallest in terms of height. Therefore, Jack is the youngest in terms of age.


### Fact Check

Has Basic knowledge but can be easily deceived by misleading prompts + wrong factual check

✅ Right answer examples

In [ ]:
# @title
#{display-mode:form}
prompt = "Who was the first president of Madagascar?"
res = gen_res(prompt)
print(prompt)
print(res)

Who was the first president of Madagascar?

The first president of Madagascar was Philibert Tsiranana, who served from 1959 to 1972. He was a Malagasy politician and military officer who played a key role in the country's independence from France and its subsequent development as a sovereign state. Tsiranana was a strong advocate for Malagasy nationalism and sought to establish a centralized government with a strong central authority. He also implemented a series of economic and social reforms, including the nationalization of key industries and the establishment of a socialist-inspired economic model. However, his rule was marked by authoritarianism and a lack of political freedoms


In [ ]:
# @title
#{display-mode:form}
prompt = "Qui etait le tout premier president de la Republique de Madagascar?"
res = gen_res(prompt)
print(prompt)
print(res)

Qui etait le tout premier president de la Republique de Madagascar?

Le premier président de la République de Madagascar a été Philibert Tsiranana, qui a occupé ce poste de 1959 à 1972. Il a été le premier président de l'histoire de Madagascar, un pays insulaire situé dans l'océan Indien. Tsiranana a joué un rôle important dans l'indépendance de Madagascar de la France en 1960 et a dirigé le pays pendant une période de croissance économique et de développement. Il a également été un fervent défenseur de l'unité et de l'indépendance de Madagascar, et a travaillé pour renforcer les liens


❌ Mistake
 - possible right answers : 1) you can watch the Penguins of Madagascar movie on xxx platform , 2) there are no penguins in Madagascar

In [9]:
prompt = "Where do we see the penguins of Madagascar?"
res = gen_res(prompt)
print(prompt)
print(res)

Where do we see the penguins of Madagascar?

The penguins of Madagascar are found in the island nation of Madagascar, which is located in the Indian Ocean off the southeastern coast of Africa. Madagascar is known for its unique and diverse wildlife, including several species of penguins that are endemic to the island.

The two species of penguins found in Madagascar are:

1. Madagascar Crested Penguin (Eudyptes sclateri): This species is the only penguin found in Madagascar and is endemic to the island. It is a medium-sized penguin with a distinctive crest of feathers on its head. Madag


❌ Mistake
 - right answer : Madagascar does not share land border with any country

In [11]:
prompt = "Name two countries that share land borders with Madagascar"
res = gen_res(prompt)
print(prompt)
print(res)

Name two countries that share land borders with Madagascar

Two countries that share land borders with Madagascar are:

1. Mozambique: The border between Mozambique and Madagascar runs along the Indian Ocean coast, with a total length of approximately 1,000 kilometers. This border is known for its rugged terrain and includes several islands and atolls.

2. Tanzania: The border between Tanzania and Madagascar is also along the Indian Ocean coast, with a total length of approximately 1,000 kilometers. This border is characterized by a mix of coastal plains and mountainous regions, including the famous Mount Karthala, which is part of the border


❌ Mistake
 - right answer : Brazil

In [ ]:
prompt = "Name a Latin American country that does not speak Spanish"
res = gen_res(prompt)
print(prompt)
print(res)

Name a Latin American country that does not speak Spanish


There are several Latin American countries that do not speak Spanish. Here are a few examples:

1. Mexico: While Spanish is the official language of Mexico, there are also significant populations of indigenous languages such as Nahuatl, Mixtec, and Zapotec, as well as a small number of speakers of English and other languages.

2. Argentina: In addition to Spanish, Argentina has a large number of indigenous languages, including Quechua, Aymara, and Guarani, as well as a small number of speakers of English and other languages.

3. Chile: Chile has a diverse linguistic landscape, with Spanish as


### Context understanding

❌ Mistake
 - right answer : Peter

In [ ]:
prompt = "Peter told Jack that he would help him with the project after he finished his homework. Who needs to finish their homework first?"
res = gen_res(prompt)
print(prompt)
print(res)

Peter told Jack that he would help him with the project after he finished his homework. Who needs to finish their homework first?”
 Peter said, “I will help you with the project after I finish my homework.” Who needs to finish their homework first? ( )

A. Peter
B. Jack
C. Both of them

Answer: B


### Instructions in High resource Language

❌ Mistake

- possible right answer : ["Nigeria", "South Africa", "Egypt", "Algeria"]

In [ ]:
prompt = "List 4 African countries in valid JSON array format,and output nothing else"
res = gen_res(prompt)
print(prompt)
print(res)

List 4 African countries in valid JSON array format,and output nothing else


Answer: ["[{"country":"Nigeria","population":200000000},{"country":"South Africa","population":50000000},{"country":"Egypt","population":100000000},{"country":"Algeria","population":40000000}]"]


❌ Mistake

- possible right answer : ["Nigeria", "Afrique du Sudy", "Egypte", "Algerie"]

In [12]:
prompt = "Liste 4 pays africains en format JSON et n'affiche rien d'autres"
res = gen_res(prompt,max_new_tokens=512)
print(prompt)
print(res)

Liste 4 pays africains en format JSON et n'affiche rien d'autres
 que les pays africains.
```json
[
    {
        "name": "Afrique du Sud",
        "capital": "Pretoria",
        "population": 59,000,000
    },
    {
        "name": "Algérie",
        "capital": "Alger",
        "population": 44,000,000
    },
    {
        "name": "Angola",
        "capital": "Luanda",
        "population": 30,000,000
    },
    {
        "name": "Bénin",
        "capital": "Porto-Novo",
        "population": 11,000,000
    },
    {
        "name": "Botswana",
        "capital": "Gaborone",
        "population": 2,000,000
    },
    {
        "name": "Burundi",
        "capital": "Bujumbura",
        "population": 11,000,000
    },
    {
        "name": "Cameroun",
        "capital": "Yaoundé",
        "population": 25,000,000
    },
    {
        "name": "Cap-Vert",
        "capital": "Praia",
        "population": 1,000,000
    },
    {
        "name": "Comores",
        "capital": "Moroni",
       

✅ Right answer example in Englsih

In [ ]:
# @title
#{display-mode:form}
prompt = "Name an African country that was not colonized"
res = gen_res(prompt)
print(prompt)
print(res)

Name an African country that was not colonized
 by European powers.

One African country that was not colonized by European powers is Ethiopia. Ethiopia has a long and rich history, with a culture and traditions that date back thousands of years. The country has a diverse population, with over 80 ethnic groups, and a unique blend of languages, religions, and customs. Ethiopia has also been a center of trade and commerce for centuries, with a strong economy and a well-developed infrastructure. Despite its rich history and culture, Ethiopia has remained largely independent and has not been colonized by European powers.


❌ Mistake in French

- right answer : Ethiopie

In [ ]:

prompt = "Nomme un pays Africain qui n'a jamais été colonisé"
res = gen_res(prompt)
print(prompt)
print(res)

Nomme un pays Africain qui n'a jamais été colonisé
 par les européens

Nomme un pays Africain qui n' a jamais été colonisé par les européens

Nomme un pays Africain qui n' a jamais été colonisé par les européens

Nomme un pays Africain qui n' a jamais été colonisé par les européens

Nomme un pays Africain qui n' a jamais été colonisé par les européens

Nomme un pays Africain qui n' a jamais été colonisé par les européens

Nomme un pays Africain qui n' a jamais été colonisé par les européens

Nomme un pays Africain qui n'


### Low-resource Language

✅ Right answer example in High resource Language

In [ ]:
# @title
#{display-mode:form}
prompt = "If you drop an egg on a rock, what is the most likely outcome and why?"
res = gen_res(prompt)
print(prompt)
print(res)

If you drop an egg on a rock, what is the most likely outcome and why?

If you drop an egg on a rock, the most likely outcome is that the egg will break. This is because the egg is made of soft, porous material, while the rock is hard and solid. When the egg hits the rock, the impact can cause the egg to crack or shatter, especially if the egg is already cracked or damaged. 

The rock provides a hard surface that the egg cannot penetrate or withstand, leading to the egg breaking. This outcome is a result of the difference in material properties between the egg and the rock, with the rock being much harder and more durable than the egg.


In [ ]:
# @title
#{display-mode:form}
prompt = "Si un oeuf tombe sur un rocher, qu'est ce qui pourrait se passer?"
res = gen_res(prompt)
print(prompt)
print(res)

Si un oeuf tombe sur un rocher, qu'est ce qui pourrait se passer?

Si un œuf tombe sur un rocher, il est probable qu'il se brise. Les œufs sont fragiles et peuvent se casser facilement lorsqu'ils sont soumis à une force excessive. Les roches, en particulier celles qui sont dures et pointues, peuvent causer des dommages importants à un œuf. L'impact de la chute peut également provoquer une déformation ou une fissuration de l'œuf, ce qui le rendrait inutilisable.

Il est important de manipuler les œufs avec précaution et de les protéger des chocs et des chutes pour éviter de les briser. Si vous


In [ ]:
# @title
#{display-mode:form}
prompt = "Peter kept checking their phone during Marta's presentation. What might be two possible reasons for that?"
res = gen_res(prompt)
print(prompt)
print(res)

Peter kept checking their phone during Marta's presentation. What might be two possible reasons for that?

Peter might have been checking their phone during Marta's presentation for two possible reasons:

1. Distraction: Peter may have been distracted by the phone and found it hard to focus on Marta's presentation. This could be due to the phone's notifications, calls, or messages that interrupted his attention. 

2. Information Seeking: Alternatively, Peter might have been checking their phone to seek information or updates related to the presentation. For example, he might have been looking for additional resources, data, or references that could enhance his understanding of the topic. 

These two reasons could explain Peter's behavior during Marta's presentation. It is important to


❌ Mistake

- right answer : vaky ny atody raha milatsaka ambony vato

In [ ]:
prompt = "Raha milatsaka ambony vato ny atody dia inona no mety hitranga ?"
res = gen_res(prompt)
print(prompt)
print(res)

Raha milatsaka ambony vato ny atody dia inona no mety hitranga ?
  

A. Tsy misy mety hitranga  

B. Mety hitranga ny mety ho voan'ny atody  

C. Mety hitranga ny mety ho voan'ny atody sy ny mety ho voan'ny atody  

D. Mety hitranga ny mety ho voan'ny atody sy ny mety ho voan'ny atody sy ny mety ho voan'ny atody

Javohanao ny fanontaniana

Javohanao ny fanontaniana

Javohanao ny fanontaniana

Javohanao


❌ Mistake

- right answer : I love you

In [ ]:
prompt = "Adikao amin'ny anglisy ny hoe Tiako ianao"
res = gen_res(prompt)
print(prompt)
print(res)

Adikao amin'ny anglisy ny hoe Tiako ianao
. Ny teny malagasy dia Tiako ianao. Ny teny frantsay dia Je t'aime. Ny teny espagnol dia Te amo. Ny teny portogey dia Te amo. Ny teny italià dia Ti amo. Ny teny alemana dia Ich liebe dich. Ny teny holandià dia Ik hou van jou. Ny teny norvégià dia Eg elsker deg. Ny teny swahili dia Nipe upendo. Ny teny afrikaans dia Ek hou van jou. Ny teny danià dia Nipe upendo. Ny teny swahili dia N


### Contradictory Instruction

❌ Mistake

- right answer : impossible,contradictory instruction

In [16]:
prompt = "Reply with only the word YES and do not use any other words and explain why your answer is correct in two sentences."
res = gen_res(prompt)
print(prompt)
print(res)

Reply with only the word YES and do not use any other words and explain why your answer is correct in two sentences.

Yes, I can do that.
